# Parallel Multiaxial Material

In [ ]:

import xara

model = xara.Model(ndm=3, ndf=3)

model.node(1, 0,0,0)
model.node(2, 1,0,0)
model.node(3, 0,1,0)
model.node(4, 0,0,1)
model.fixZ(0.0,  1,1,1)

E1 = 1000.0
E2 = 2000.0
W1 = 0.5
W2 = 0.5

model.nDMaterial("ElasticIsotropic", 100, E1, 0.0)
model.nDMaterial("ElasticIsotropic", 200, E2, 0.0)
model.nDMaterial("Parallel3D", 300,  (100,200), weights=(W1, W2))
model.element("FourNodeTetrahedron", 1,  (1,2,3,4),  300)

# Loads
model.timeSeries("Constant", 1)
model.pattern("Plain", 1, 1)
model.load(4, (0.0, 0.0, 1.0/6.0)) # sigma33=1



In [ ]:

model.constraints("Transformation")
model.numberer("RCM")
model.system("FullGeneral")
model.test("NormDispIncr", 1.0e-5, 100, 0)
model.algorithm("Newton")
model.integrator("LoadControl", 1.0)
model.analysis("Static")
model.analyze(1)

0

In [ ]:
# the stress computed by the parallel material
S0_33 = model.eleResponse(1, "material", 1, "stress")[2] 

# manually homogenized from sub-materials, should, be equal to S0_33
S0_33_homogenized = model.eleResponse(1, "material", 1, "homogenized", "stress")[2] 
S1_33 = model.eleResponse(1, "material", 1,    "material", 1, "stress")[2] # S33 from the first material
S2_33 = model.eleResponse(1, "material", 1,    "material", 2, "stress")[2] # S33 from the first material

print("   Parallel Sigma33 = {:.3g}".format(S0_33))
print("Homogenized Sigma33 = {:.3g} (must be equal to the previous)".format(S0_33_homogenized))
print(" Sub-mat (1)Sigma33 = {:.3g}".format(S1_33))
print(" Sub-mat (2)Sigma33 = {:.3g}".format(S2_33))
print("Check:")
print("(1)Sigma33*W1 + (2)Sigma33*W2 = Sigma33")
print("{:.3g}*{:.3g} + {:.3g}*{:.3g} = {:.3g}".format(S1_33,W1, S2_33,W2, S1_33*W1+S2_33*W2))

   Parallel Sigma33 = 1
Homogenized Sigma33 = 1 (must be equal to the previous)
 Sub-mat (1)Sigma33 = 0.667
 Sub-mat (2)Sigma33 = 1.33
Check:
(1)Sigma33*W1 + (2)Sigma33*W2 = Sigma33
0.667*0.5 + 1.33*0.5 = 1
